***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
pd.options.display.float_format = '{:.1f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_main = os.path.join(path_sp, 'Data')
    path_out  = os.path.join(path_main, 'Next Gen of Mobility Solutions', 'Road')

path_code    = os.path.join(path_git, 'Data', 'EIA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://www.eia.gov/opendata/documentation.php
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

Road_1

***

In [ ]:
root_ = 'https://api.eia.gov/v2'

cat = 'petroleum'
cat_ = f'/{cat}'

route = 'pri/gnd'
route_ = f'/{route}'

freq = 'annual'
value = 'value'
geography = 'SCA'
data_ = f'/data/?frequency={freq}&data[0]={value}&facets[duoarea][]={geography}'

api_key_ = f'&api_key={api_key}'

    
query = f"{root_}{cat_}{route_}{data_}{api_key_}"

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_road1_1 = pd.concat(list_df)
df_road1_1 = df_road1_1.sort_values('period')
df_road1_1 = df_road1_1.reset_index(drop = True)

df_road1_1 = df_road1_1[df_road1_1['product-name'] == 'Regular Gasoline']
df_road1_1.head()

In [ ]:
root_ = 'https://api.eia.gov/v2'

cat = 'petroleum'
cat_ = f'/{cat}'

route = 'pri/gnd'
route_ = f'/{route}'

freq = 'annual'
value = 'value'
geography = 'NUS'
data_ = f'/data/?frequency={freq}&data[0]={value}&facets[duoarea][]={geography}'

api_key_ = f'&api_key={api_key}'

    
query = f"{root_}{cat_}{route_}{data_}{api_key_}"

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_road1_2 = pd.concat(list_df)
df_road1_2 = df_road1_2.sort_values('period')
df_road1_2 = df_road1_2.reset_index(drop = True)

df_road1_2 = df_road1_2[df_road1_2['product-name'] == 'Regular Gasoline']
df_road1_2.head()

In [ ]:
df_road1 = pd.concat([df_road1_1, df_road1_2])

df_road1 = df_road1.dropna()
df_road1 = df_road1[df_road1['value'] != 'null']

df_road1 = df_road1.sort_values(['period', 'duoarea'], ascending = [False, True])
df_road1 = df_road1.reset_index(drop = True)
df_road1 = df_road1[['period', 'area-name', 'product', 'product-name', 'process-name', 'value']]
df_road1.columns = ['Year', 'Geography', 'Product', 'Product Name', 'Process', 'Price']
df_road1['Price'] = df_road1['Price'].astype('float32')
df_road1['Year' ] = df_road1['Year' ].astype('int')
df_road1 = df_road1[df_road1['Year'] >= 2000]

df_road1.head()

In [ ]:
year_start = df_road1.Year.min()
year_end   = df_road1.Year.max()

df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_' + str(year_end)]]

df_road1 = df_road1.merge(df_cpi, on = 'Year', how = 'left')
df_road1['Price'] = round(df_road1['Price']*df_road1['IAF_' + str(year_end)], 2)
df_road1 = df_road1.drop(['IAF_' + str(year_end)], axis = 1)


df_road1.loc[df_road1['Geography'] == 'U.S.'      , 'Geography'] = 'National'
df_road1.loc[df_road1['Geography'] == 'CALIFORNIA', 'Geography'] = 'California'


df_road1.head()

In [ ]:
indicator_name = 'Road_1'
sample_type = 'EIA'
tag = 'Gas Prices'



df_about = write_about(sample_type      = sample_type
                       , indicator_name = indicator_name
                       , year_start     = year_start
                       , year_end       = year_end
                       , path_config0   = path_config0)

df_about

In [ ]:

workbook_name = f'{indicator_name} {tag} {sample_type}.xlsx'

with pd.ExcelWriter(os.path.join(path_out, workbook_name), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About')
    df_road1.to_excel(writer, index = False, sheet_name = 'Gas Prices')

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data"

workbook_name = f'{indicator_name} {tag} {sample_type}.xlsx'


with pd.ExcelWriter(os.path.join(path_plots, workbook_name), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About')
    df_road1.to_excel(writer, index = False, sheet_name = 'Gas Prices')

print('Export Location: ' + path_plots)
